# SignSync: live demo on Colab
Runs the SignSync web app from GitHub and opens it in a browser tab through Colab, so a video can be uploaded and translated from any device.

Run the cells **in order**. The first run downloads about 3.5 GB into your Google Drive (allow a few minutes). Later runs reuse it and are much faster.
Use *Runtime > Change runtime type > T4 GPU* first if it is offered. The link stops working when this notebook stops, so keep the tab open while presenting.

Model code and weights are Uni-Sign's (Li et al., 2025, CC BY-NC 4.0, non-commercial use); dataset is How2Sign (Duarte et al., 2021).

## 1. Get the app code from GitHub

In [ ]:
import os
os.chdir('/content')
if not os.path.isdir('SignSync-asl-translation'):
    !git clone https://github.com/Kashish811/SignSync-asl-translation
os.chdir('/content/SignSync-asl-translation')
!ls

## 2. Install the requirements
Torch is already on Colab, so it is skipped here. If a message asks you to restart the runtime, do it, then run from cell 1 again.

In [ ]:
!grep -v -E "torch|extra-index-url" requirements.txt > /tmp/req_colab.txt
!pip install -q -r /tmp/req_colab.txt
!pip install -q rtmlib

## 3. Get Uni-Sign code, the How2Sign checkpoint and mT5-base (cached in your Google Drive)
The first run downloads about 3.5 GB into a Drive folder called `signsync_cache`. Later runs reuse that folder and take only a minute or two. Allow Drive access when asked.

In [ ]:
import os, subprocess
from google.colab import drive
from huggingface_hub import hf_hub_download, snapshot_download
drive.mount('/content/drive')

cache = '/content/drive/MyDrive/signsync_cache'
os.makedirs(cache, exist_ok=True)
os.chdir('/content/SignSync-asl-translation')

hf_hub_download('ZechengLi19/Uni-Sign', 'how2sign_pose_only_slt.pth', local_dir=cache + '/hf_unisign')
snapshot_download('google/mt5-base', local_dir=cache + '/mt5-base',
                  ignore_patterns=['*.h5', 'flax_model*', '*.msgpack', 'tf_model*', '*.ot'])

if not os.path.isdir('Uni-Sign'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ZechengLi19/Uni-Sign'], check=True)
os.makedirs('Uni-Sign/pretrained_weight', exist_ok=True)
for link, target in [('hf_unisign', cache + '/hf_unisign'), ('Uni-Sign/pretrained_weight/mt5-base', cache + '/mt5-base')]:
    if not os.path.lexists(link):
        os.symlink(target, link)
print('model assets ready')

## 4. Start the app

In [ ]:
import subprocess, time, os
os.environ['UNISIGN_DIR'] = os.path.abspath('Uni-Sign')
os.environ['CKPT'] = os.path.abspath('hf_unisign/how2sign_pose_only_slt.pth')
server = subprocess.Popen(['python', '-m', 'uvicorn', 'main:app', '--app-dir', 'app', '--host', '127.0.0.1', '--port', '8000'],
                          stdout=open('/tmp/uvicorn.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(8)
!curl -s http://127.0.0.1:8000/api/health

## 5. Open the app
**Use this on the college Wi-Fi.** The cell below opens the app in a new browser tab through Colab's own address. It works in this browser only (the one signed in to Colab). Allow pop-ups for colab.research.google.com if the tab does not open.

If the tab is blocked, click the link the cell prints instead.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8000)

### Optional: a public link for other devices
Only works on networks that do not block `trycloudflare.com`. The college Wi-Fi may block it.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
import re
link = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        link = m.group(0); break
print('OPEN THIS LINK:', link)

## 6. Keep it alive
Leave this tab open. If the app stops, run cell 4 again, then cell 5.
To read the app log if something fails, run `!tail -30 /tmp/uvicorn.log` in a new cell.